In [ ]:
%pip install -q azure-ai-ml azure-identity

In [ ]:
# Cell 1: setup - run once
%pip install -q huggingface_hub azureml-fsspec adlfs azure-identity azure-ai-ml azure-storage-blob datasets

from huggingface_hub import HfApi, login
# paste your HF write token - get from https://huggingface.co/settings/tokens
# need WRITE access for esthxy/rlvr_group_correlation
login(token="hf_xxxxxxxxxxx") 

api = HfApi()
print("Logged in as:", api.whoami()["name"])

In [ ]:
import os

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)
print(os.getcwd())


In [ ]:
!python submit_stage1.py --step env

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Initialize the ML Client with subscription ID, resource group, and workspace name
ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)

# Loop through environments to find and print the latest version details
for n in ["rlvr-stage1-cpu", "rlvr-stage1-gpu"]:
    v = max(int(e.version) for e in ml_client.environments.list(name=n))
    e = ml_client.environments.get(n, version=str(v))
    print(n, "v" + str(v), "|", e.creation_context.created_at)


In [ ]:
import sys,os

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)
print(sys.executable)

In [ ]:
!{sys.executable} submit_stage1.py --step env

In [ ]:
import glob
import json
import os
import sys
import time

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def watch(job_name, poll=60):
    """Poll instead of stream — survives notebook disconnects."""
    while True:
        s = ml_client.jobs.get(job_name).status
        print(time.strftime("%H:%M:%S"), job_name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


In [ ]:
for n in ["rlvr-stage1-cpu", "rlvr-stage1-gpu"]:
    vs = [int(e.version) for e in ml_client.environments.list(name=n)]
    print(n, "latest v" + str(max(vs)))


In [ ]:
!{sys.executable} submit_stage1.py --step compute

In [ ]:
c = ml_client.compute.get("t4v3")
c.min_instances = 0
c.idle_time_before_scale_down = 900

ml_client.compute.begin_create_or_update(c).result()

print("t4v3 min_instances =", ml_client.compute.get("t4v3").min_instances)


In [ ]:
!{sys.executable} submit_stage1.py --step prompts

In [ ]:
JOB_PROMPTS = "PASTE_JOB_NAME_FROM_ABOVE"
watch(JOB_PROMPTS)

In [ ]:
JOB_PROMPTS = submit("01_build_prompts.yml")
watch(JOB_PROMPTS)
logs(JOB_PROMPTS, "[prompts]")

In [ ]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)


In [ ]:
JOB_PROMPTS = submit("01 build prompts.yml")
watch(JOB_PROMPTS)
logs(JOB_PROMPTS, "[prompts]")

In [ ]:
JOB_PILOT = submit("02 pilot generate.yml")
watch(JOB_PILOT)
logs(JOB_PILOT, "[gen]")

In [ ]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=120):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for l in open(f, errors="ignore"):
            if needle in l:
                print(l.rstrip())


In [ ]:
ml_client.environments.create_or_update(
    Environment(
        name="rlvr-stage1-cpu-torch",
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
        conda_file=os.path.join(PROJ, "env/stage1-cpu-torch.yml"),
    )
)
print("registered")


In [ ]:
JOB_PILOT = submit("02b_pilot_generate_cpu.yml")
watch(JOB_PILOT, poll=180)
logs(JOB_PILOT, "[cpu-gen]")

In [ ]:
from azure.ai.ml.entities import Data
ml_client.data.create_or_update(Data(
name="tmp-pilot-list", version="1", type="uri_folder",
path="azureml://datastores/workspaceblobstore/paths/rlvr_group_corr/rollouts_pilot/"))

In [ ]:
JOB_GATE = submit("03 pilot gate.yml")
watch(JOB_GATE)

In [ ]:
# ============================================================
# STAGE 1 — GATE (job 03) : full runnable cell
# Re-run this whole cell after any kernel restart.
# ============================================================
import ast
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=30):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


# --- 1. verify the new pilot_gate.py parses locally (catches paste errors) ---
ast.parse(open(f"{PROJ}/src/pilot_gate.py").read())
print("pilot_gate.py syntax OK")

# --- 2. write the gate job YAML (scoped to the 0.5B pilot only) -------------
gate_yaml = """$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
type: command
display_name: stage1-03c-PILOT-GATE
experiment_name: rlvr-group-correlation

code: ../src
command: >-
  python pilot_gate.py
  --rollouts ${{inputs.rollouts}}
  --out ${{outputs.gate}}/gate_report.json
  --only_model qwen2.5-0.5b-cpu-pilot
  --min_mixed 0.30
  --max_cat_share 0.98
  --min_disagree 0.02

environment: azureml:rlvr-stage1-cpu@latest
compute: azureml:cpu-audit-cluster

inputs:
  rollouts:
    type: uri_folder
    mode: ro_mount
    path: azureml://datastores/workspaceblobstore/paths/rlvr_group_corr/rollouts_pilot/

outputs:
  gate:
    type: uri_folder
    mode: rw_mount
    path: azureml://datastores/workspaceblobstore/paths/rlvr_group_corr/gate/
"""
open(f"{PROJ}/jobs/03c_pilot_gate.yml", "w").write(gate_yaml)
print("wrote jobs/03c_pilot_gate.yml")

# --- 3. submit and wait ----------------------------------------------------
JOB_GATE = submit("03c_pilot_gate.yml")
status = watch(JOB_GATE)
print("\njob status:", status, " (Failed may mean a GATE failed, exit 2)")

# --- 4. print the log so gate lines are visible either way ------------------
d = f"./_logs_{JOB_GATE}"
ml_client.jobs.download(JOB_GATE, download_path=d, all=True)
for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
    for line in open(f, errors="ignore"):
        if any(
            t in line
            for t in (
                "[gate]",
                "GATE ",
                "DO NOT PROCEED",
                "All gates clear",
                "Error",
                "Traceback",
            )
        ):
            print(line.rstrip())

# --- 5. read the report ----------------------------------------------------
try:
    ml_client.jobs.download(
        JOB_GATE, download_path="./_gate", output_name="gate"
    )
    r = json.load(
        open(glob.glob("./_gate/**/gate_report.json", recursive=True)[0])
    )

    print("\n" + "=" * 64)
    print(
        f"files {r['n_files']} | groups {r['n_groups']} | models {r['models_present']}"
    )
    print(
        f"G1 mixed-group rate {r['gate1_mixed_group_rate']:<8} >= {r['gate1_threshold']} "
        f"{'PASS' if r['gate1_pass'] else 'FAIL'}"
    )
    print(
        f"G2 mean cat share {r['gate2_mean_primary_cat_share']:<8} < {r['gate2_threshold_max_share']} "
        f"{'PASS' if r['gate2_pass'] else 'FAIL'}"
    )
    print(
        f"G3 disagreement rate {r['gate3_verifier_disagreement_rate']:<8} >= {r['gate3_threshold']} "
        f"{'PASS' if r['gate3_pass'] else 'FAIL'}"
    )
    print(f"\nALL GATES PASS: {r['ALL_GATES_PASS']}")
    print(
        f"prelim rho_cat {r['preliminary_rho_cat_proxy']} CI95 {r['preliminary_rho_ci95']}"
    )
    print(f"Kish n_eff / k {r['kish_n_eff_of_k']} of {r['k_bar']}")
    print("\nrho by primary category:")
    print(json.dumps(r["rho_by_primary_cat"], indent=2))
    print("\ncategory distribution:")
    print(json.dumps(r["primary_cat_distribution"], indent=2))
    print("=" * 64)
except Exception as e:
    print("\ncould not read gate_report.json:", type(e).__name__, e)


In [ ]:
print(ml_client.compute.get("t4v3").provisioning_state) # need Succeeded + free
JOB_04 = submit("04 full.yml")
print(JOB_04)

In [ ]:
# ============================================================
# STAGE 1 — GATE (job 03) : full runnable cell
# Re-run this whole cell after any kernel restart.
# ============================================================
import ast
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=30):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)

JOB_GATE = submit("03d_gate_real.yml")
watch(JOB_GATE)

In [ ]:
# ============================================================
# STAGE 1 — GATE (job 03) : full runnable cell
# Re-run this whole cell after any kernel restart.
# ============================================================
import ast
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=30):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)
ml_client.jobs.download(JOB_04, download_path="./_out_1p5b", output_name="rollouts")
import glob, json
for f in glob.glob("./_out_1p5b/**/_summary_qwen2.5-1.5b.json", recursive=True):print(json.load(open(f)))

In [ ]:
JOB_05 = submit("05_full_generate_7b_awq.yml")
print(JOB_05)

In [ ]:
JOB_04 = submit("04_full_generate_1p5b.yml")
ml_client.jobs.download(JOB_04, download_path="./_out_1p5b", output_name="rollouts")
import glob, json
for f in glob.glob("./_out_1p5b/**/_summary_qwen2.5-1.5b.json", recursive=True):print(json.load(open(f)))

In [ ]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)


JOB_VERIFY = submit("06_rule_verify_1p5b.yml")
watch(JOB_VERIFY, poll=60)

In [ ]:
JOB_REPLAY = submit("07_advantage_replay_1p5b.yml")
watch(JOB_REPLAY, poll=30)

In [ ]:
ml_client.jobs.download(JOB_REPLAY, download_path="./_replay", output_name="report")
r = json.load(open(glob.glob("./_replay/**/replay_report_1p5b.json", recursive=True)[0]))
print(json.dumps(r, indent=2))

In [ ]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)

JOB_GOLD = submit("08_gold_sampling_1p5b.yml")
watch(JOB_GOLD, poll=30)

In [ ]:
# after job 05 has ~4-8 shards
JOB_GATE_7B = submit("03e_gate_7b.yml")
watch(JOB_GATE_7B)

In [ ]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)

JOB_SELECT = submit("09_select_compass_sample.yml")
watch(JOB_SELECT, poll=30)


In [ ]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())
JOB_COMPASS = submit("10_compass_verify.yml")
watch(JOB_COMPASS, poll=120)

In [ ]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "",
    "",
    "",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())
# 3. once job 10 shows Completed, run final analysis
JOB_FINAL = submit("11_final_analysis.yml")
watch(JOB_FINAL, poll=30)

In [ ]:
# 1. gate on what's there now, before stopping
JOB_GATE_7B = submit("03e_gate_7b.yml")
watch(JOB_GATE_7B)